In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath("../src"))

## 1. Bype Pair Encoding (BPE)

In [ ]:
from collections import defaultdict

corpus = {
    "low": 5,
    "lower": 2,
    "lowest": 6,
    "newer": 3,
}

vocab = {" ".join(list(mot) + ["</w>"]): freq for mot, freq in corpus.items()}
print("Vocabulaire initial (caracteres seuls) :")
for mot, freq in vocab.items():
    print(f"  '{mot}' (freq={freq})")


def get_pair_counts(vocab):
    pairs = defaultdict(int)
    for mot, freq in vocab.items():
        symboles = mot.split()
        for i in range(len(symboles) - 1):
            pairs[(symboles[i], symboles[i + 1])] += freq
    return pairs


def merge_pair(pair, vocab):
    nouveau_vocab = {}
    bigram = " ".join(pair)
    remplacement = "".join(pair)
    for mot in vocab:
        nouveau_mot = mot.replace(bigram, remplacement)
        nouveau_vocab[nouveau_mot] = vocab[mot]
    return nouveau_vocab


print()
print("=== Fusions successives ===")
for etape in range(5):
    paires = get_pair_counts(vocab)
    meilleure_paire = max(paires, key=paires.get)
    print(
        f"Etape {etape + 1} : fusion de {meilleure_paire} \
            (vue {paires[meilleure_paire]} fois) -> '{''.join(meilleure_paire)}'"
    )
    vocab = merge_pair(meilleure_paire, vocab)

print()
print("Vocabulaire apres 5 fusions :")
for mot, freq in vocab.items():
    print(f"  '{mot}' (freq={freq})")

Vocabulaire initial (caracteres seuls) :
  'l o w </w>' (freq=5)
  'l o w e r </w>' (freq=2)
  'l o w e s t </w>' (freq=6)
  'n e w e r </w>' (freq=3)

=== Fusions successives ===
Etape 1 : fusion de ('l', 'o') (vue 13             fois) -> 'lo'
Etape 2 : fusion de ('lo', 'w') (vue 13             fois) -> 'low'
Etape 3 : fusion de ('low', 'e') (vue 8             fois) -> 'lowe'
Etape 4 : fusion de ('lowe', 's') (vue 6             fois) -> 'lowes'
Etape 5 : fusion de ('lowes', 't') (vue 6             fois) -> 'lowest'

Vocabulaire apres 5 fusions :
  'low </w>' (freq=5)
  'lowe r </w>' (freq=2)
  'lowest </w>' (freq=6)
  'n e w e r </w>' (freq=3)


In [5]:
from tokenization.bpe import tokenize_with_bpe, train_bpe_tokenizer

corpus = [
    "the delivery was fast",
    "the delivery was slow",
    "the shipping was fast",
    "the shipping was slow",
    "the product quality was great",
    "the product quality was poor",
    "customer service was helpful",
    "customer service was rude",
    "the packaging was damaged",
    "the packaging was excellent",
] * 20

tokenizer = train_bpe_tokenizer(corpus, vocab_size=100, min_frequency=2)
print("\nTaille du vocabulaire appris :", tokenizer.get_vocab_size())





Taille du vocabulaire appris : 100


In [6]:
# --- BLOC 3 : mot connu vs mot jamais vu vs mot totalement etranger ---
print("'delivery' (mot connu) :", tokenize_with_bpe(tokenizer, "delivery"))
print(
    "'deliveryman' (jamais vu, fragment connu) :",
    tokenize_with_bpe(tokenizer, "deliveryman"),
)
print("'xyzabc123' (aucun rapport) :", tokenize_with_bpe(tokenizer, "xyzabc123"))


# --- BLOC 4 : effet de vocab_size sur le niveau de decoupe ---
for taille in [20, 50, 200]:
    t = train_bpe_tokenizer(corpus, vocab_size=taille, min_frequency=2)
    tokens = tokenize_with_bpe(t, "delivery")
    print(
        f"vocab_size={taille:4} -> 'delivery' decoupe en : {tokens} (vocabulaire reel \
            : {t.get_vocab_size()})"
    )

'delivery' (mot connu) : ['delivery']
'deliveryman' (jamais vu, fragment connu) : ['delivery', 'm', 'a', 'n']
'xyzabc123' (aucun rapport) : ['x', 'y', '[UNK]', 'a', '[UNK]', 'c', '[UNK]', '[UNK]', '[UNK]']



vocab_size=  20 -> 'delivery' decoupe en : ['d', 'e', 'l', 'i', 'v', 'e', 'r', 'y'] (vocabulaire reel             : 24)



vocab_size=  50 -> 'delivery' decoupe en : ['de', 'li', 'v', 'er', 'y'] (vocabulaire reel             : 50)



vocab_size= 200 -> 'delivery' decoupe en : ['delivery'] (vocabulaire reel             : 101)


## . Wordpiece

In [8]:
corpus_jouet = {"low": 5, "lower": 2, "lowest": 6, "newer": 3}
vocab = {" ".join(list(mot) + ["</w>"]): freq for mot, freq in corpus_jouet.items()}


def get_symbol_freqs(vocab):
    freqs = defaultdict(int)
    for mot, freq in vocab.items():
        for symbole in mot.split():
            freqs[symbole] += freq
    return freqs


def get_pair_counts(vocab):
    pairs = defaultdict(int)
    for mot, freq in vocab.items():
        symboles = mot.split()
        for i in range(len(symboles) - 1):
            pairs[(symboles[i], symboles[i + 1])] += freq
    return pairs


pair_counts = get_pair_counts(vocab)
symbol_freqs = get_symbol_freqs(vocab)

resultats = []
for paire, freq_paire in pair_counts.items():
    a, b = paire
    score_bpe = freq_paire
    score_wp = freq_paire / (symbol_freqs[a] * symbol_freqs[b])
    resultats.append((paire, freq_paire, score_bpe, score_wp))

print("--- Top 3 selon BPE (frequence brute) ---")
for paire, fp, s_bpe, s_wp in sorted(resultats, key=lambda x: -x[2])[:3]:
    print(f"  {paire} : freq={fp}, score_BPE={s_bpe}, score_WordPiece={s_wp:.4f}")

print("\n--- Top 3 selon WordPiece (score normalise) ---")
for paire, fp, s_bpe, s_wp in sorted(resultats, key=lambda x: -x[3])[:3]:
    print(f"  {paire} : freq={fp}, score_BPE={s_bpe}, score_WordPiece={s_wp:.4f}")

--- Top 3 selon BPE (frequence brute) ---
  ('l', 'o') : freq=13, score_BPE=13, score_WordPiece=0.0769
  ('o', 'w') : freq=13, score_BPE=13, score_WordPiece=0.0625
  ('w', 'e') : freq=11, score_BPE=11, score_WordPiece=0.0491

--- Top 3 selon WordPiece (score normalise) ---
  ('s', 't') : freq=6, score_BPE=6, score_WordPiece=0.1667
  ('l', 'o') : freq=13, score_BPE=13, score_WordPiece=0.0769
  ('e', 'r') : freq=5, score_BPE=5, score_WordPiece=0.0714


In [9]:
# --- BLOC 2 : version bibliotheque -- BPE vs WordPiece cote a cote ---

from tokenization.bpe import tokenize_with_bpe, train_bpe_tokenizer
from tokenization.wordpiece import tokenize_with_wordpiece, train_wordpiece_tokenizer

corpus = [
    "the delivery was fast",
    "the delivery was slow",
    "the shipping was fast",
    "the shipping was slow",
    "the product quality was great",
    "the product quality was poor",
    "customer service was helpful",
    "customer service was rude",
    "the packaging was damaged",
    "the packaging was excellent",
] * 20

tok_bpe = train_bpe_tokenizer(corpus, vocab_size=60, min_frequency=2)
tok_wp = train_wordpiece_tokenizer(corpus, vocab_size=60, min_frequency=2)


# --- BLOC 3 : comparaison sur plusieurs mots, avec attention au prefixe "##" ---
mots_test = ["delivery", "deliveryman", "packaging", "unpackaged"]

for mot in mots_test:
    print(f"'{mot}':")
    print("  BPE      :", tokenize_with_bpe(tok_bpe, mot))
    print("  WordPiece:", tokenize_with_wordpiece(tok_wp, mot))







'delivery':
  BPE      : ['de', 'liver', 'y']
  WordPiece: ['del', '##i', '##v', '##er', '##y']
'deliveryman':
  BPE      : ['de', 'liver', 'y', 'm', 'a', 'n']
  WordPiece: ['del', '##i', '##v', '##er', '##y', '##m', '##a', '##n']
'packaging':
  BPE      : ['pac', 'kag', 'ing']
  WordPiece: ['pa', '##c', '##k', '##ag', '##ing']
'unpackaged':
  BPE      : ['u', 'n', 'pac', 'kag', 'e', 'd']
  WordPiece: ['u', '##n', '##p', '##a', '##c', '##k', '##ag', '##e', '##d']


## 3. SentencePiece

In [11]:
import os
import sys
import tempfile

sys.path.insert(0, os.path.abspath("../../src"))

from tokenization.sentencepiece import (
    detokenize_with_sentencepiece,
    tokenize_with_sentencepiece,
    train_sentencepiece_tokenizer,
)

corpus_lignes = [
    "the delivery was fast",
    "the delivery was slow",
    "la livraison etait rapide",
    "la livraison etait lente",
    "डिलीवरी बहुत तेज़ थी",
    "डिलीवरी बहुत धीमी थी",
    "the product quality was great",
    "la qualite du produit est excellente",
    "उत्पाद की गुणवत्ता उत्कृष्ट है",
] * 30

tmp_dir = tempfile.mkdtemp()
corpus_file = os.path.join(tmp_dir, "corpus.txt")
with open(corpus_file, "w", encoding="utf-8") as f:
    f.write("\n".join(corpus_lignes))
print("Fichier corpus cree ici :", corpus_file)

Fichier corpus cree ici : /tmp/tmp1_4zjkpx/corpus.txt


In [12]:
# --- BLOC 2 : entrainement ---
processor = train_sentencepiece_tokenizer(
    corpus_file_path=corpus_file,
    model_prefix=os.path.join(tmp_dir, "sp_model"),
    vocab_size=200,
    model_type="bpe",
    character_coverage=1.0,  # important pour bien couvrir le devanagari
)


# --- BLOC 3 : le MEME code sur 3 langues/scripts differents ---
for texte in [
    "the delivery was fast",
    "la livraison etait rapide",
    "डिलीवरी बहुत तेज़ थी",
]:
    tokens = tokenize_with_sentencepiece(processor, texte)
    reconstruit = detokenize_with_sentencepiece(processor, tokens)
    print(f"Texte       : {texte}")
    print(f"Tokens      : {tokens}")
    print(f"Reconstruit : {reconstruit}")
    print(f"Identique   : {texte == reconstruit}")
    print()

Texte       : the delivery was fast
Tokens      : ['▁the', '▁delivery', '▁was', '▁fast']
Reconstruit : the delivery was fast
Identique   : True

Texte       : la livraison etait rapide
Tokens      : ['▁la', '▁livraison', '▁etait', '▁rapide']
Reconstruit : la livraison etait rapide
Identique   : True

Texte       : डिलीवरी बहुत तेज़ थी
Tokens      : ['▁डिलीवरी', '▁बहुत', '▁तेज़', '▁थी']
Reconstruit : डिलीवरी बहुत तेज़ थी
Identique   : True



I0000 00:00:1785412844.529452 2021364 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: /tmp/tmp1_4zjkpx/corpus.txt
  input_format: 
  model_prefix: /tmp/tmp1_4zjkpx/sp_model
  model_type: BPE
  vocab_size: 200
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_su

In [ ]:
# --- BLOC 4 : pourquoi la reversibilite compte -- espacement irregulier ---
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

tok_bpe = Tokenizer(BPE(unk_token="[UNK]"))
tok_bpe.pre_tokenizer = Whitespace()
tok_bpe.train_from_iterator(
    ["the delivery was fast"] * 20, BpeTrainer(vocab_size=50, special_tokens=["[UNK]"])
)

texte_irregulier = "the delivery was fast"
tokens_bpe = tok_bpe.encode(texte_irregulier).tokens
reconstruit_bpe_naif = " ".join(tokens_bpe)

tokens_sp = tokenize_with_sentencepiece(processor, texte_irregulier)
reconstruit_sp = detokenize_with_sentencepiece(processor, tokens_sp)

print(
    "BPE - reconstruction naive :",
    repr(reconstruit_bpe_naif),
    "identique :",
    reconstruit_bpe_naif == texte_irregulier,
)
print(
    "SentencePiece - reconstruction :",
    repr(reconstruit_sp),
    "identique :",
    reconstruit_sp == texte_irregulier,
)



BPE - reconstruction naive : 'the delivery was fast' identique : True
SentencePiece - reconstruction : 'the delivery was fast' identique : True

